# Interpretable Rule Learning in PyTorch

**Course:** Interpretable Machine Learning — IIT Kharagpur  
**Group 11**

| Member | Roll | Contribution |
|---|---|---|
| Aritra Bose | 25EC65R08 | Data loading, preprocessing, and baselines |
| Chunarkar Sujeeth Kumar | 24AI10021 | Sequential Covering |
| Mohammad Asad ur Rahman Nisar | 24AI10038 | Bayesian Rule List |
| Kotikala Yashwant | 24AI10034 | Evaluation, visualisation, and integration |

This notebook is a runnable front end for the submitted modular implementation.
The implementations live in `src/`; importing them here keeps the notebook,
command-line pipeline, report, and saved results on the same code path.

## Setup

Run this notebook from the repository root after installing `requirements.txt`.
Stages 3, 4, and 6 use seed 42. Stage 5 reports seeds 42, 43, and 44.

In [ ]:
from pathlib import Path
import math
import os
import random

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import numpy as np
import pandas as pd
import torch
from IPython.display import Image, display
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score,
                             roc_curve)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import CategoricalNB

DATA_PATH = "data/adult.data"
RESULTS_DIR = Path("results")
FIGURES_DIR = RESULTS_DIR / "figures"
SEED = 42

print(f"PyTorch: {torch.__version__}")
print(f"NumPy: {np.__version__}")
print(f"pandas: {pd.__version__}")

## Implementation — data preprocessing and literal encoding

This cell contains the implementation used by the notebook. It is intentionally
self-contained: no local project module is imported.

In [ ]:
COLUMNS = [
    "age", "workclass", "fnlwgt", "education", "education-num",
    "marital-status", "occupation", "relationship", "race", "sex",
    "capital-gain", "capital-loss", "hours-per-week", "native-country", "income",
]
AGE_BINS, AGE_LABELS = [0, 25, 45, 65, 120], [
    "Young (<=25)", "Middle-aged (26-45)", "Senior-Adult (46-65)", "Senior (>65)"]
HOURS_BINS, HOURS_LABELS = [0, 34, 40, 55, 120], [
    "Part-time (<35)", "Full-time (35-40)", "Overtime (41-55)", "Extreme (>55)"]
GAIN_BINS, GAIN_LABELS = [-1, 0, 7000, 1_000_000], ["None", "Moderate (<7k)", "High (>7k)"]
LOSS_BINS, LOSS_LABELS = [-1, 0, 1800, 1_000_000], ["None", "Low-Loss (<=1.8k)", "High-Loss (>1.8k)"]


def load_adult(file_path=DATA_PATH, verbose=True):
    """Load Adult data, remove missing rows, and encode >50K as 1."""
    df = pd.read_csv(file_path, names=COLUMNS, sep=r",\s*", engine="python",
                     na_values=["?", " ?"])
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].str.strip()  # preserves NaN for dropna
    n_before = len(df)
    df = df.dropna().reset_index(drop=True)
    df["income"] = df["income"].astype(str).str.rstrip(".")
    y = (df["income"] == ">50K").astype(int).values
    X = df.drop(columns=["income"])
    if verbose:
        print(f"Rows before dropna : {n_before}")
        print(f"Rows after  dropna : {len(df)}")
        print(f"Class balance      : <=50K {1-y.mean():.2%} | >50K {y.mean():.2%}")
    return X, y


def discretise(df):
    """Produce readable categorical features for the rule learners."""
    df = df.copy().drop(columns=["fnlwgt", "education-num"])
    df["age_group"] = pd.cut(df["age"].astype(float), bins=AGE_BINS,
                               labels=AGE_LABELS, right=True).astype(str)
    df["hours_group"] = pd.cut(df["hours-per-week"].astype(float), bins=HOURS_BINS,
                                 labels=HOURS_LABELS, right=True).astype(str)
    df["capital_gain_group"] = pd.cut(df["capital-gain"].astype(float), bins=GAIN_BINS,
                                        labels=GAIN_LABELS, right=True).astype(str)
    df["capital_loss_group"] = pd.cut(df["capital-loss"].astype(float), bins=LOSS_BINS,
                                        labels=LOSS_LABELS, right=True).astype(str)
    df = df.drop(columns=["age", "hours-per-week", "capital-gain", "capital-loss"])
    df["native_country_group"] = np.where(df["native-country"] == "United-States",
                                            "United-States", "Other")
    return df.drop(columns=["native-country"])


class LiteralEncoder:
    """Encode each (column, value) proposition as a boolean PyTorch row."""
    def fit(self, X_df):
        self.columns_ = list(X_df.columns)
        self.literals_ = [(col, value) for col in self.columns_ for value in pd.unique(X_df[col])]
        self.col_index_ = {col: i for i, col in enumerate(self.columns_)}
        self.literal_col_ = torch.tensor([self.col_index_[col] for col, _ in self.literals_],
                                         dtype=torch.long)
        return self

    def encode(self, X_df):
        return torch.from_numpy(np.stack([X_df[col].values == value
                                          for col, value in self.literals_]))

    def rule_mask(self, M, literal_idx):
        return torch.ones(M.shape[1], dtype=torch.bool) if not literal_idx else M[list(literal_idx)].all(0)

    def text(self, literal_idx):
        return " AND ".join(f"[{col} = '{value}']" for col, value in
                            (self.literals_[i] for i in literal_idx))

## Implementation — Sequential Covering

In [ ]:
class SequentialCovering:
    def __init__(self, max_rules=10, max_conds=2, prune_fraction=0.33,
                 beam_width=1, random_state=42):
        self.max_rules, self.max_conds = max_rules, max_conds
        self.prune_fraction, self.beam_width = prune_fraction, beam_width
        self.random_state = random_state

    def _grow(self, M, y, enc):
        covered, rule = torch.ones(M.shape[1], dtype=torch.bool), []
        while len(rule) < self.max_conds:
            p0, n0 = int((covered & y).sum()), int((covered & ~y).sum())
            if p0 == 0 or n0 == 0:
                break
            C = M & covered
            p1, n1 = (C & y).sum(1).double(), (C & ~y).sum(1).double()
            prec0 = p0 / (p0 + n0 + 1e-9)
            prec1 = p1 / (p1 + n1 + 1e-9)
            gain = p1 * (torch.log2(prec1.clamp_min(1e-12)) - math.log2(prec0))
            gain[(p1 == 0) | (prec1 <= prec0)] = -1.0
            if rule:
                gain[rule] = -1.0
            best = int(torch.argmax(gain))
            if float(gain[best]) <= 0:
                break
            rule.append(best)
            covered = C[best]
        return rule

    def _cohen(self, M, y, lits, enc):
        mask = enc.rule_mask(M, lits)
        p, n = int((mask & y).sum()), int((mask & ~y).sum())
        return (-1.0 if p + n == 0 else (p - n) / (p + n)), p, n

    def _prune(self, M, y, rule, enc):
        best_s, best_p, best_n = self._cohen(M, y, rule, enc)
        best_rule = list(rule)
        for k in range(len(rule) - 1, 0, -1):
            score, p, n = self._cohen(M, y, rule[:k], enc)
            if score >= best_s:
                best_s, best_p, best_n, best_rule = score, p, n, rule[:k]
        return best_rule if best_p > best_n else None

    def fit(self, X_df, y, enc):
        generator = torch.Generator().manual_seed(self.random_state)
        M, y = enc.encode(X_df), torch.from_numpy(np.asarray(y).astype(bool))
        alive = torch.ones(M.shape[1], dtype=torch.bool)
        self.rules_, self.stats_ = [], []
        while len(self.rules_) < self.max_rules:
            pos, neg = torch.nonzero(alive & y).squeeze(1), torch.nonzero(alive & ~y).squeeze(1)
            if len(pos) == 0 or len(neg) == 0:
                break
            pos, neg = pos[torch.randperm(len(pos), generator=generator)], neg[torch.randperm(len(neg), generator=generator)]
            pp, nn = max(1, int(len(pos) * self.prune_fraction)), max(1, int(len(neg) * self.prune_fraction))
            prune_idx, grow_idx = torch.cat([pos[:pp], neg[:nn]]), torch.cat([pos[pp:], neg[nn:]])
            grown = self._grow(M[:, grow_idx], y[grow_idx], enc)
            if not grown:
                break
            pruned = self._prune(M[:, prune_idx], y[prune_idx], grown, enc)
            if not pruned:
                break
            covered = enc.rule_mask(M, pruned) & alive
            p, n = int((covered & y).sum()), int((covered & ~y).sum())
            if p == 0:
                break
            self.rules_.append(pruned)
            self.stats_.append({"p": p, "n": n, "precision": p / (p + n)})
            alive &= ~covered
        self.default_prob_ = float((alive & y).sum()) / int(alive.sum()) if alive.any() else 0.0
        self._enc = enc
        return self

    def predict_proba(self, X_df):
        M = self._enc.encode(X_df)
        prob, uncovered = torch.full((M.shape[1],), self.default_prob_, dtype=torch.double), torch.ones(M.shape[1], dtype=torch.bool)
        for literals, stats in zip(self.rules_, self.stats_):
            hit = self._enc.rule_mask(M, literals) & uncovered
            prob[hit], uncovered = stats["precision"], uncovered & ~hit
        return prob.numpy()

    def print_rules(self, enc):
        print(f"\nSequential Covering rules ({len(self.rules_)}):")
        for i, (literals, stats) in enumerate(zip(self.rules_, self.stats_), 1):
            print(f"  Rule {i}: IF {enc.text(literals)} THEN >50K (precision {stats['precision']:.3f}, covers {stats['p'] + stats['n']} [pos {stats['p']}, neg {stats['n']}])")
        print(f"  DEFAULT: ELSE <=50K (residual positive rate {self.default_prob_:.3f})\n")

## Implementation — Bayesian Rule List

In [ ]:
class BayesianRuleList:
    def __init__(self, alpha0=1.0, alpha1=1.0, list_prior_lambda=3.0,
                 width_penalty=0.5, max_len=8, minsupport=0.03,
                 n_iter=4000, random_state=42):
        self.alpha0, self.alpha1 = alpha0, alpha1
        self.lam, self.width_penalty = list_prior_lambda, width_penalty
        self.max_len, self.minsupport = max_len, minsupport
        self.n_iter, self.random_state = n_iter, random_state

    def _mine(self, M, enc, min_count):
        counts = M.sum(1)
        singletons = torch.nonzero(counts >= min_count).squeeze(1)
        selected = M[singletons].float()
        joint = selected @ selected.T
        columns = enc.literal_col_[singletons]
        valid = torch.triu((joint >= min_count) & (columns[:, None] != columns[None, :]), diagonal=1)
        pi, pj = torch.nonzero(valid, as_tuple=True)
        self.cand_ = [(enc.literals_[int(i)],) for i in singletons]
        masks = [M[singletons]]
        self.cand_ += [(enc.literals_[int(singletons[a])], enc.literals_[int(singletons[b])])
                       for a, b in zip(pi.tolist(), pj.tolist())]
        if len(pi):
            masks.append(M[singletons[pi]] & M[singletons[pj]])
        return torch.cat(masks, 0)

    def _log_leaf(self, n0, n1):
        a0, a1 = self.alpha0, self.alpha1
        return (math.lgamma(a0 + a1) - math.lgamma(a0) - math.lgamma(a1)
                + math.lgamma(n0 + a0) + math.lgamma(n1 + a1)
                - math.lgamma(n0 + n1 + a0 + a1))

    def _log_posterior(self, order, candidate_masks, y):
        covered, log_likelihood, probabilities, counts = torch.zeros(candidate_masks.shape[1], dtype=torch.bool), 0.0, [], []
        for index in order:
            mask = candidate_masks[index] & ~covered
            n1, n0 = int((mask & y).sum()), int((mask & ~y).sum())
            counts.append((n0, n1))
            probabilities.append((n1 + self.alpha1) / (n0 + n1 + self.alpha0 + self.alpha1))
            log_likelihood += self._log_leaf(n0, n1)
            covered |= mask
        default = ~covered
        n1, n0 = int((default & y).sum()), int((default & ~y).sum())
        counts.append((n0, n1))
        probabilities.append((n1 + self.alpha1) / (n0 + n1 + self.alpha0 + self.alpha1))
        log_likelihood += self._log_leaf(n0, n1)
        log_prior = len(order) * math.log(self.lam) - math.lgamma(len(order) + 1)
        log_prior -= sum(self.width_penalty * (len(self.cand_[index]) - 1) for index in order)
        return log_likelihood + log_prior, probabilities, counts

    def _mh(self, candidate_masks, y, seed):
        rng, n_candidates = random.Random(seed), candidate_masks.shape[0]
        best_single, best_score = 0, -1e18
        for candidate in range(n_candidates):
            score, _, _ = self._log_posterior([candidate], candidate_masks, y)
            if score > best_score:
                best_single, best_score = candidate, score
        current = [best_single]
        current_score, current_prob, current_counts = self._log_posterior(current, candidate_masks, y)
        best = (list(current), current_score, current_prob, current_counts)

        def other(excluded):
            for _ in range(50):
                candidate = rng.randrange(n_candidates)
                if candidate not in excluded:
                    return candidate
            return None

        for _ in range(self.n_iter):
            move, proposal = rng.randrange(4), list(current)
            q_forward = q_backward = 0.0
            current_set = set(current)
            if move == 0 and len(proposal) < self.max_len:  # add
                candidate = other(current_set)
                if candidate is None:
                    continue
                forward_candidates, forward_positions = max(n_candidates - len(current_set), 1), len(current) + 1
                proposal.insert(rng.randint(0, len(proposal)), candidate)
                q_forward, q_backward = 1 / (forward_candidates * forward_positions), 1 / len(proposal)
            elif move == 1 and len(proposal) > 1:  # remove
                proposal.pop(rng.randrange(len(proposal)))
                proposal_set = set(proposal)
                q_forward = 1 / len(current)
                q_backward = 1 / (max(n_candidates - len(proposal_set), 1) * (len(proposal) + 1))
            elif move == 2 and len(proposal) > 1:  # swap
                i, j = rng.sample(range(len(proposal)), 2)
                proposal[i], proposal[j] = proposal[j], proposal[i]
                q_forward = q_backward = 1.0
            elif move == 3:  # replace
                index, candidate = rng.randrange(len(proposal)), other(current_set)
                if candidate is None:
                    continue
                proposal[index] = candidate
                proposal_set = set(proposal)
                q_forward = 1 / (len(current) * max(n_candidates - len(current_set), 1))
                q_backward = 1 / (len(proposal) * max(n_candidates - len(proposal_set), 1))
            else:
                continue
            proposal_score, proposal_prob, proposal_counts = self._log_posterior(proposal, candidate_masks, y)
            log_alpha = proposal_score - current_score + math.log(q_backward) - math.log(q_forward)
            if math.log(rng.random() + 1e-12) < log_alpha:
                current, current_score, current_prob, current_counts = proposal, proposal_score, proposal_prob, proposal_counts
                if current_score > best[1]:
                    best = (list(current), current_score, current_prob, current_counts)
        return best

    def fit(self, X_df, y, enc):
        self._enc = enc
        M, target = enc.encode(X_df), torch.from_numpy(np.asarray(y).astype(bool))
        self.cand_masks_ = self._mine(M, enc, int(self.minsupport * M.shape[1]))
        self.best_order_, self.best_score_, self.leaf_probs_, self.leaf_counts_ = self._mh(self.cand_masks_, target, self.random_state)
        return self

    def predict_proba(self, X_df):
        M = self._enc.encode(X_df)
        probabilities, uncovered = torch.full((M.shape[1],), self.leaf_probs_[-1], dtype=torch.double), torch.ones(M.shape[1], dtype=torch.bool)
        for candidate, probability in zip(self.best_order_, self.leaf_probs_[:-1]):
            literals = [self._enc.literals_.index(literal) for literal in self.cand_[candidate]]
            hit = self._enc.rule_mask(M, literals) & uncovered
            probabilities[hit], uncovered = probability, uncovered & ~hit
        return probabilities.numpy()

    def print_rules(self):
        print(f"\nBayesian Rule List ({len(self.best_order_)} rules + DEFAULT):")
        for i, (candidate, probability, counts) in enumerate(zip(self.best_order_, self.leaf_probs_[:-1], self.leaf_counts_[:-1]), 1):
            condition = " AND ".join(f"[{col} = '{value}']" for col, value in self.cand_[candidate])
            print(f"  Rule {i}: IF {condition} THEN P(>50K) = {probability:.3f} (n0={counts[0]}, n1={counts[1]})")
        print(f"  DEFAULT: ELSE P(>50K) = {self.leaf_probs_[-1]:.3f} (n0={self.leaf_counts_[-1][0]}, n1={self.leaf_counts_[-1][1]})\n")

## Implementation — evaluation and visualisation helpers

In [ ]:
def metrics(y, probability, threshold):
    prediction = (probability >= threshold).astype(int)
    return dict(acc=accuracy_score(y, prediction),
                prec=precision_score(y, prediction, zero_division=0),
                rec=recall_score(y, prediction, zero_division=0),
                f1=f1_score(y, prediction, zero_division=0),
                auc=roc_auc_score(y, probability))


def best_tau(y_val, probability):
    """Select the minority-class F1 threshold from validation data only."""
    threshold, score = 0.5, -1.0
    for candidate in np.unique(np.round(probability, 4)):
        candidate_score = f1_score(y_val, (probability >= candidate).astype(int), zero_division=0)
        if candidate_score > score:
            threshold, score = float(candidate), candidate_score
    return threshold


def evaluate(file_path=DATA_PATH, seeds=(42, 43, 44), brl_iter=4000, out_dir="results"):
    os.makedirs(out_dir, exist_ok=True)
    X, y = load_adult(file_path, verbose=False)
    X = discretise(X)
    rows = []
    for seed in seeds:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=seed, stratify=y)
        X_sub, X_val, y_sub, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=seed, stratify=y_train)
        encoder = LiteralEncoder().fit(X_sub)
        models = [("SeqCovering", SequentialCovering(random_state=seed).fit(X_sub, y_sub, encoder),
                   lambda model: (len(model.rules_), sum(len(rule) for rule in model.rules_))),
                  ("BRL", BayesianRuleList(n_iter=brl_iter, random_state=seed).fit(X_sub, y_sub, encoder),
                   lambda model: (len(model.best_order_), sum(len(model.cand_[i]) for i in model.best_order_)))]
        for name, model, size in models:
            val_prob, test_prob = model.predict_proba(X_val), model.predict_proba(X_test)
            threshold = best_tau(y_val, val_prob)
            rules, literals = size(model)
            fixed, tuned = metrics(y_test, test_prob, 0.5), metrics(y_test, test_prob, threshold)
            row = dict(model=name, seed=seed, rules=rules, literals=literals, tau=threshold)
            row.update({f"fixed_{key}": value for key, value in fixed.items()})
            row.update({f"tuned_{key}": value for key, value in tuned.items()})
            rows.append(row)
    results = pd.DataFrame(rows)
    results.to_csv(Path(out_dir) / "evaluation_raw.csv", index=False)
    print("=== Per-run results (test set) ===")
    print(results[["model", "seed", "rules", "literals", "tau", "fixed_acc", "fixed_f1", "fixed_auc", "tuned_f1", "tuned_rec"]].round(4).to_string(index=False))
    print("\n=== Mean +/- std over seeds ===")
    summary = results.groupby("model").agg(["mean", "std"])
    for model in ("SeqCovering", "BRL"):
        values = summary.loc[model]
        print(f"\n{model}:")
        for label, key in [("rules", "rules"), ("literals", "literals"), ("accuracy", "fixed_acc"),
                           ("precision", "fixed_prec"), ("recall", "fixed_rec"),
                           ("F1 (0.5)", "fixed_f1"), ("F1 (tuned)", "tuned_f1"), ("AUC", "fixed_auc")]:
            print(f"  {label:10}: {values[(key, 'mean')]:.4f} +/- {values[(key, 'std')]:.4f}")
    points = results.groupby("model").agg(literals=("literals", "mean"), f1=("tuned_f1", "mean"))
    fig, ax = plt.subplots(figsize=(6, 4.5))
    for name, point in points.iterrows():
        ax.scatter(point["literals"], point["f1"], s=90, marker="o" if name == "SeqCovering" else "s")
        ax.annotate(name, (point["literals"], point["f1"]), fontsize=9)
    ax.set(xlabel="Explanation size (total literals)", ylabel="Test F1 (>50K), tuned on validation",
           title="Accuracy vs complexity")
    fig.tight_layout()
    fig.savefig(Path(out_dir) / "evaluation_pareto.png", dpi=150)
    plt.close(fig)
    return results


def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def _box(ax, x, y, width, height, text, color):
    ax.add_patch(FancyBboxPatch((x, y), width, height, boxstyle="round,pad=0.02",
                                linewidth=1.2, edgecolor="black", facecolor=color))
    ax.text(x + width / 2, y + height / 2, text, ha="center", va="center", fontsize=9, wrap=True)


def plot_decision_list(rows, title, path):
    n = len(rows)
    fig, ax = plt.subplots(figsize=(12, max(3, 1.05 * n + 1)))
    ax.set(xlim=(0, 1), ylim=(0, n + 0.5)); ax.axis("off")
    for i, (condition, outcome) in enumerate(rows):
        y = n - i - 0.5
        default = i == n - 1
        _box(ax, 0.06, y, 0.70, 0.70, ("ELSE" if default else f"Rule {i+1}") + "\n" + condition,
             "#f3f4f6" if default else ("#dbeafe" if i % 2 == 0 else "#fee2e2"))
        ax.text(0.80, y + 0.35, outcome, ha="left", va="center", fontsize=9)
    ax.set_title(title, fontsize=12, fontweight="bold")
    fig.tight_layout(); fig.savefig(path, dpi=140, bbox_inches="tight"); plt.close(fig)


def decision_list_rows_sc(model, encoder):
    rows = [(f"IF {encoder.text(literals)}", f"P(>50K) = {stats['precision']:.3f}")
            for literals, stats in zip(model.rules_, model.stats_)]
    return rows + [(f"residual P(>50K) = {model.default_prob_:.3f}", "DEFAULT")]


def decision_list_rows_brl(model):
    rows = [("IF " + " AND ".join(f"{column} = {value}" for column, value in model.cand_[candidate]),
             f"P(>50K) = {probability:.3f}")
            for candidate, probability in zip(model.best_order_, model.leaf_probs_[:-1])]
    return rows + [(f"P(>50K) = {model.leaf_probs_[-1]:.3f}", "DEFAULT")]


def plot_roc_curves(curves, path):
    fig, ax = plt.subplots(figsize=(6, 5.5))
    for name, target, probability in curves:
        fpr, tpr, _ = roc_curve(target, probability)
        ax.plot(fpr, tpr, linewidth=2, label=f"{name} (AUC = {roc_auc_score(target, probability):.3f})")
    ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Chance")
    ax.set(xlabel="False positive rate", ylabel="True positive rate (>50K)", title="ROC curves on the held-out test set")
    ax.legend(loc="lower right", fontsize=9); ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(path, dpi=140); plt.close(fig)


def plot_confusion_matrices(models, target, threshold, path):
    fig, axes = plt.subplots(1, len(models), figsize=(5 * len(models), 4.6))
    for ax, (name, probability) in zip(np.atleast_1d(axes), models):
        matrix = confusion_matrix(target, (probability >= threshold).astype(int))
        ax.imshow(matrix, cmap="Blues"); ax.set_title(f"{name} (threshold {threshold:.2f})")
        ax.set(xlabel="Predicted", ylabel="Actual", xticks=[0, 1], yticks=[0, 1],
               xticklabels=["<=50K", ">50K"], yticklabels=["<=50K", ">50K"])
        for i in range(2):
            for j in range(2):
                ax.text(j, i, str(matrix[i, j]), ha="center", va="center", fontsize=13,
                        fontweight="bold", color="white" if matrix[i, j] > matrix.max() / 2 else "black")
    fig.tight_layout(); fig.savefig(path, dpi=140); plt.close(fig)


def plot_calibration(models, target, path, n_bins=8):
    fig, ax = plt.subplots(figsize=(6, 5.5)); ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
    bins = np.linspace(0, 1, n_bins + 1)
    for name, probability in models:
        indices = np.digitize(probability, bins) - 1
        means, observed = [], []
        for bin_index in range(n_bins):
            selected = indices == bin_index
            if selected.sum() >= 10:
                means.append(probability[selected].mean()); observed.append(target[selected].mean())
        ax.plot(means, observed, "o-", linewidth=2, markersize=6, label=name)
    ax.set(xlabel="Mean predicted P(>50K)", ylabel="Observed frequency of >50K", title="Calibration / reliability diagram")
    ax.legend(loc="upper left", fontsize=9); ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(path, dpi=140); plt.close(fig)

---
## Stage 1 — Data loading and baselines

`load_adult` treats `?` as missing, strips whitespace, removes incomplete
rows, and converts `>50K` to the positive label. The categorical Naive Bayes
baseline is included for comparison only; both rule learners are implemented
in this repository.

In [ ]:
X_raw, y = load_adult(DATA_PATH)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.20, random_state=SEED, stratify=y
)

majority_pred = np.zeros_like(y_test)
print("Majority-class baseline")
print(f"  Accuracy : {accuracy_score(y_test, majority_pred):.4f}")
print(f"  Precision: {precision_score(y_test, majority_pred, zero_division=0):.4f}")
print(f"  Recall   : {recall_score(y_test, majority_pred, zero_division=0):.4f}")
print(f"  F1       : {f1_score(y_test, majority_pred, zero_division=0):.4f}")


def fit_codes(frame):
    return {column: {value: i for i, value in enumerate(pd.unique(frame[column]))}
            for column in frame.columns}


def apply_codes(frame, codes):
    encoded = np.zeros((len(frame), len(frame.columns)), dtype=int)
    for column_index, column in enumerate(frame.columns):
        encoded[:, column_index] = [codes[column].get(value, 0)
                                    for value in frame[column]]
    return encoded


codes = fit_codes(X_train_raw)
naive_bayes = CategoricalNB().fit(apply_codes(X_train_raw, codes), y_train)
nb_prob = naive_bayes.predict_proba(apply_codes(X_test_raw, codes))[:, 1]
nb_pred = (nb_prob >= 0.5).astype(int)
print("\nCategorical Naive Bayes baseline")
print(f"  Accuracy : {accuracy_score(y_test, nb_pred):.4f}")
print(f"  Precision: {precision_score(y_test, nb_pred, zero_division=0):.4f}")
print(f"  Recall   : {recall_score(y_test, nb_pred, zero_division=0):.4f}")
print(f"  F1       : {f1_score(y_test, nb_pred, zero_division=0):.4f}")
print(f"  AUC      : {roc_auc_score(y_test, nb_prob):.4f}")

---
## Stage 2 — Discretisation and PyTorch literal encoding

Continuous values are converted to readable bins and the redundant `fnlwgt`
and `education-num` columns are removed. `LiteralEncoder` represents every
proposition as a boolean PyTorch coverage row: `M[literal, example]`.

In [ ]:
X_discrete = discretise(X_raw)
encoder_all = LiteralEncoder().fit(X_discrete)
M_all = encoder_all.encode(X_discrete)
print(f"Features after discretisation: {X_raw.shape[1]} -> {X_discrete.shape[1]}")
print(f"Total literals: {len(encoder_all.literals_)}")
print(f"Coverage tensor M: {tuple(M_all.shape)}, dtype={M_all.dtype}")

for column in X_discrete.columns:
    literal_rows = [i for i, (name, _) in enumerate(encoder_all.literals_)
                    if name == column]
    assert bool(torch.all(M_all[literal_rows].sum(0) == 1))
print("Sanity check passed: each example satisfies exactly one literal per feature.")

---
## Stage 3 — Sequential Covering

The model grows rules using FOIL gain, prunes them with Cohen's metric, and
removes every training example covered by an accepted rule. A locally seeded
PyTorch generator controls the grow/prune split.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_discrete, y, test_size=0.20, random_state=SEED, stratify=y
)
encoder = LiteralEncoder().fit(X_train)
sequential_covering = SequentialCovering(random_state=SEED).fit(
    X_train, y_train, encoder
)
sequential_covering.print_rules(encoder)

sc_prob = sequential_covering.predict_proba(X_test)
sc_pred = (sc_prob >= 0.5).astype(int)
print("Sequential Covering test metrics")
print(f"  Accuracy : {accuracy_score(y_test, sc_pred):.4f}")
print(f"  Precision: {precision_score(y_test, sc_pred, zero_division=0):.4f}")
print(f"  Recall   : {recall_score(y_test, sc_pred, zero_division=0):.4f}")
print(f"  F1       : {f1_score(y_test, sc_pred, zero_division=0):.4f}")
print(f"  AUC      : {roc_auc_score(y_test, sc_prob):.4f}")

---
## Stage 4 — Bayesian Rule List

The BRL pre-mines frequent one- and two-literal antecedents, scores ordered
lists using a Dirichlet-multinomial marginal likelihood plus a length prior,
and explores lists through seeded Metropolis-Hastings moves.

In [ ]:
bayesian_rule_list = BayesianRuleList(random_state=SEED).fit(
    X_train, y_train, encoder
)
bayesian_rule_list.print_rules()

brl_prob = bayesian_rule_list.predict_proba(X_test)
brl_pred = (brl_prob >= 0.5).astype(int)
print("Bayesian Rule List test metrics")
print(f"  Accuracy : {accuracy_score(y_test, brl_pred):.4f}")
print(f"  Precision: {precision_score(y_test, brl_pred, zero_division=0):.4f}")
print(f"  Recall   : {recall_score(y_test, brl_pred, zero_division=0):.4f}")
print(f"  F1       : {f1_score(y_test, brl_pred, zero_division=0):.4f}")
print(f"  AUC      : {roc_auc_score(y_test, brl_prob):.4f}")

---
## Stage 5 — Held-out multi-seed evaluation

A 60/20/20 train/validation/test split is run for seeds 42, 43, and 44.
The threshold is selected on validation F1 only, then scored on held-out test
data. This writes `results/evaluation_raw.csv` and the Pareto figure.

In [ ]:
evaluate(DATA_PATH, seeds=(42, 43, 44), brl_iter=4000, out_dir=str(RESULTS_DIR))

---
## Stage 6 — Visualisation

Generate the report figures from the fixed seed-42 80/20 split.

In [ ]:
ensure_dir(str(FIGURES_DIR))
discrete_codes = {
    column: {value: i for i, value in enumerate(pd.unique(X_train[column]))}
    for column in X_train.columns
}


def encode_discrete(frame):
    return np.asarray([
        [discrete_codes[column].get(value, 0) for column, value in row.items()]
        for _, row in frame.iterrows()
    ], dtype=int)


nb_discrete = CategoricalNB().fit(encode_discrete(X_train), y_train)
nb_discrete_prob = nb_discrete.predict_proba(encode_discrete(X_test))[:, 1]

plot_decision_list(
    decision_list_rows_sc(sequential_covering, encoder),
    "Sequential Covering decision list",
    str(FIGURES_DIR / "sequential_covering_decision_list.png"),
)
plot_decision_list(
    decision_list_rows_brl(bayesian_rule_list),
    "Bayesian Rule List decision list",
    str(FIGURES_DIR / "bayesian_rule_list_decision_list.png"),
)
plot_roc_curves(
    [("Sequential Covering", y_test, sc_prob),
     ("Bayesian Rule List", y_test, brl_prob),
     ("Naive Bayes", y_test, nb_discrete_prob)],
    str(FIGURES_DIR / "roc_curves.png"),
)
plot_confusion_matrices(
    [("Sequential Covering", sc_prob), ("Bayesian Rule List", brl_prob)],
    y_test, 0.5, str(FIGURES_DIR / "confusion_matrices.png"),
)
plot_calibration(
    [("Sequential Covering", sc_prob), ("Bayesian Rule List", brl_prob)],
    y_test, str(FIGURES_DIR / "calibration.png"),
)

print("Saved figures:")
for figure in sorted(FIGURES_DIR.glob("*.png")):
    print(f"  {figure}")

### Generated figures

In [ ]:
for name in (
    "sequential_covering_decision_list.png",
    "bayesian_rule_list_decision_list.png",
    "roc_curves.png",
    "confusion_matrices.png",
    "calibration.png",
):
    display(Image(filename=str(FIGURES_DIR / name)))